In [1]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Gene Exp & Drug Response 

In [2]:
drug_response = pd.read_csv("../gdsc1_data/drugAct.csv", index_col=0)
gene_exp = pd.concat(
    [
        pd.read_csv("../gdsc1_data/gene_exp_part1.csv.gz", index_col=0).T,
        pd.read_csv("../gdsc1_data/gene_exp_part2.csv.gz", index_col=0).T,
    ],
    axis=1,
).dropna()
cells = sorted(
    set(drug_response.columns)
    & set(gene_exp.index)
    & set(pd.read_csv("../gdsc1_data/mut.csv", index_col=0).T.index)
)

In [3]:
# def get_smiles_from_compound_name(compound_name, max_retries=5, delay=5):
#     # First try to get SMILES from local file
#     try:
#         df = pd.read_csv(
#             "../Figs/nsc_cid_smiles_class_name.csv", usecols=["NAME", "SMILES"]
#         )
#         match = df[df["NAME"].str.lower() == compound_name.lower()]
#         if not match.empty:
#             return match.iloc[0]["SMILES"]
#     except Exception as e:
#         print(f"Error reading local SMILES data: {e}")

#     # If not found locally, try PubChem API
#     url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{compound_name}/property/CanonicalSMILES/JSON"

#     for attempt in range(max_retries):
#         try:
#             response = requests.get(url)
#             response.raise_for_status()
#             data = response.json()
#             smiles = (
#                 data.get("PropertyTable", {})
#                 .get("Properties", [{}])[0]
#                 .get("CanonicalSMILES")
#             )
#             return smiles
#         except requests.RequestException as e:
#             if response.status_code == 503 or "PUGREST.ServerBusy" in str(e):
#                 print(
#                     f"Server busy. Retrying in {delay} seconds... (Attempt {attempt + 1}/{max_retries})"
#                 )
#                 time.sleep(delay)
#                 delay *= 2  # Exponential backoff
#             else:
#                 print(f"Error retrieving SMILES for compound name {compound_name}: {e}")
#                 return None

#     print(f"Failed to retrieve SMILES after {max_retries} attempts")
#     return None


# def process_compound(compound_name):
#     return [get_smiles_from_compound_name(compound_name), compound_name]


# def get_smiles_parallel(compound_names, max_workers=10):
#     results = []
#     with ThreadPoolExecutor(max_workers=max_workers) as executor:
#         futures = [executor.submit(process_compound, name) for name in compound_names]
#         for future in tqdm(
#             as_completed(futures),
#             total=len(compound_names),
#             desc="Processing compounds",
#         ):
#             results.append(future.result())
#     return results

In [4]:
# compound_names = drug_response.index.tolist()
# SMILES = get_smiles_parallel(compound_names)

In [5]:
# SMILES = pd.DataFrame(SMILES)
# SMILES

In [6]:
# missing = [
#     "965-D2",
#     "993-D2",
#     "AKT inhibitor VIII",
#     "AKT inhibitor VIII",
#     "AZD4547",
#     "AZD4547",
#     "AZD6482",
#     "AZD6482",
#     "AZD7762",
#     "AZD7762",
#     "AZD7969",
#     "Afatinib",
#     "Afatinib",
#     "Avagacestat",
#     "Avagacestat",
#     "BAY ACCi",
#     "BAY AKT1",
#     "BAY-HDAC11",
#     "BAY-HDAC11",
#     "BAY-HDAC11",
#     "paclitaxel",
#     "paclitaxel",
#     "BAY-MPS1",
#     "BMS-536924",
#     "BMS-536924",
#     "Bicalutamide",
#     "Bicalutamide",
#     "Bleomycin",
#     "Bleomycin",
#     "Brivanib",
#     "CAP-232",
#     "CCT245467",
#     "CHIR-99021",
#     "CHIR-99021",
#     "CRT0105446",
#     "CRT0160829",
#     "Cetuximab",
#     "Cisplatin",
#     "Cisplatin",
#     "Doxorubicin",
#     "Doxorubicin",
#     "Dyrk1b",
#     "EphB4",
#     "FEN1",
#     "FGFR",
#     "FGFR",
#     "FS106",
#     "FS112",
#     "FY012",
#     "FY026",
#     "FY069",
#     "GSK269962A",
#     "GSK269962A",
#     "Gemcitabine",
#     "Gemcitabine",
#     "Genentech Cpd 10",
#     "HG-6-71-01",
#     "IAP_5620",
#     "IAP_7638",
#     "ICL1100013",
#     "IGFR_3801",
#     "JAK1_3715",
#     "JAK3_7406",
#     "JNK-9L",
#     "JQ1_1218",
#     "JQ1_163",
#     "JW-7-52-1",
#     "KIN001-042",
#     "KIN001-236",
#     "KIN001-260",
#     "MCT1_6447",
#     "MCT4_1422",
#     "MPS-1-IN-1",
#     "A832234",
#     "N22899-6-C1",
#     "N23918-95-7",
#     "N24798-49-A1",
#     "NG-25_1167",
#     "NG-25_260",
#     "Nutlin-3a (-)",
#     "Olaparib",
#     "Olaparib",
#     "PARP_0108",
#     "PARP_9482",
#     "PARP_9495",
#     "PI3Ka_4409",
#     "PLK_6522",
#     "PLX-4720",
#     "PLX-4720",
#     "Pictilisib",
#     "Pictilisib",
#     "QL-VIII-58",
#     "QL-XII-47",
#     "QL-XII-47",
#     "QL-XII-61",
#     "RAF_9304",
#     "Refametinib",
#     "Refametinib",
#     "SB505124",
#     "SB505124",
#     "SN-38",
#     "SN-38",
#     "Selumetinib",
#     "Selumetinib",
#     "TANK",
#     "THZ-1-87",
#     "THZ-2-102-1",
#     "THZ-2-49",
#     "THZ-2-98-01",
#     "TL-1-85",
#     "TL-2-105",
#     "TTK",
#     "UNC0638",
#     "UNC0638",
#     "VNLG/124",
#     "Venotoclax",
#     "XMD11-50",
#     "XMD15-27",
#     "ZL049",
#     "ZL109",
#     "A-484954",
#     "rTRAIL",
# ]
# missing_SMILES = get_smiles_parallel(missing)

In [7]:
# k = pd.DataFrame([missing, sorted(list(SMILES[SMILES[0].isna()][1]))]).T
# k.columns = [1, 2]

In [8]:
# tmp = pd.DataFrame(missing_SMILES).merge(k).drop_duplicates().drop(1, axis=1).dropna()
# tmp.columns = ["SMILES", "drugs"]
# SMILES = SMILES.dropna()
# SMILES.columns = ["SMILES", "drugs"]
# SMILES = pd.concat([tmp, SMILES])

In [9]:
# SMILES.to_csv('gdsc1_drug2smiles.csv')

In [10]:
SMILES = pd.read_csv("gdsc1_drug2smiles.csv", index_col=0)
SMILES

,SMILES,drugs
0,C1CN(CCC1N2C3=CC=CC=C3NC2=O)CC4=CC=C(C=C4)C5=C...,AKT inhibitor VIII_171
1,C1CN(CCC1N2C3=CC=CC=C3NC2=O)CC4=CC=C(C=C4)C5=C...,AKT inhibitor VIII_228
2,C1CC(CNC1)NC(=O)C2=C(C=C(S2)C3=CC(=CC=C3)F)NC(...,AZD7762_1022
3,C1CC(CNC1)NC(=O)C2=C(C=C(S2)C3=CC(=CC=C3)F)NC(...,AZD7762_1402
4,CC1=CN2C(=O)C=C(N=C2C(=C1)C(C)NC3=CC=CC=C3C(=O...,AZD6482_1066
...,...,...
372,C1CC(CN(C1)C2=C(C=CC=C2C3=CC=CC=C3)C=C4C(=O)NC...,AZD1208
379,CC(C)OC1=NNC(=C1)NC2=NC(=NC=C2Cl)NC(C)C3=NC=C(...,AZD1332
390,C1CC2=CC=CC=C2C1NC3=C4C=CN(C4=NC=N3)C5CC(C(C5)...,Pevonedistat
391,C1C2CN(C1CN2C3=CC=CC=N3)C=CC(=O)C4=CC=CC=C4O,PFI-3


In [11]:
gene_exp = gene_exp.loc[cells]
gene_exp

Gene,A1BG,A1CF,A2M,A2ML1,A3GALT2,A3GALT2P,A4GALT,A4GNT,AAAS,AACS,...,ZW10,ZWILCH,ZWINT,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
22RV1,3.537942,6.364651,5.332441,2.923030,2.815383,2.815383,3.241125,3.262633,4.722157,4.942126,...,8.378382,7.526319,9.274166,3.785757,5.201929,2.742575,4.635015,3.940190,4.658663,8.088672
23132-87,3.370950,6.284884,3.485675,2.831562,2.913369,2.913369,3.313028,3.096527,4.873621,4.213177,...,8.586287,6.526713,8.021952,3.520584,4.957371,2.636933,4.350120,4.395806,4.598583,7.951146
42-MG-BA,6.016743,3.191894,3.249480,2.808959,2.626195,2.626195,3.063389,3.356853,4.919962,4.978849,...,8.574633,7.274847,7.683935,3.473062,5.237112,2.759740,5.517990,4.391955,4.157708,8.944875
451Lu,5.725238,2.971953,10.768500,2.928547,2.750283,2.750283,3.367116,3.162653,5.144177,4.242312,...,7.805859,7.942236,9.385336,3.292793,5.530937,2.582158,5.087147,5.197606,4.922167,7.470603
5637,2.927335,2.892365,3.181651,2.926549,2.677943,2.677943,4.295357,3.205598,5.249042,4.495021,...,8.498743,7.057572,9.261486,3.492602,4.813163,2.558705,4.580977,4.810975,4.371501,7.974367
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YT,3.143593,3.779792,5.095163,2.879333,2.749091,2.749091,3.186225,3.359589,4.511721,4.241912,...,8.467307,8.196766,9.205680,3.515323,5.152341,2.472749,5.046783,6.016707,5.168183,7.063949
ZR-75-30,5.594177,3.032126,3.562195,3.108549,2.767673,2.767673,3.611381,3.247737,4.547910,4.423134,...,7.725739,7.398583,8.781244,3.574957,5.301549,3.045222,4.952083,4.604161,4.472996,7.675603
huH-1,4.177894,10.272718,5.050040,2.904163,3.162503,3.162503,3.205621,3.332665,5.467095,3.694510,...,8.486763,6.799690,8.755107,3.602078,5.365176,2.990428,4.844390,4.089650,5.042040,8.574266
no-10,4.106684,3.229351,3.070295,2.793178,2.806316,2.806316,3.203713,3.365893,4.312271,4.227046,...,8.861172,7.475805,8.647136,3.569171,5.473651,2.534410,4.967557,4.798076,4.447893,8.113379


In [12]:
drug_response = drug_response.loc[sorted(SMILES.drugs), cells]
drug_response

,22RV1,23132-87,42-MG-BA,451Lu,5637,639-V,647-V,697,769-P,786-0,...,WSU-DLCL2,WSU-NHL,YAPC,YH-13,YKG-1,YT,ZR-75-30,huH-1,no-10,no-11
(5Z)-7-Oxozeaenol,-1.074268,-0.565177,-0.255724,1.441198,-0.706426,-0.341020,-0.247842,0.262330,-0.265945,-0.334071,...,-1.567757,0.035238,-1.205127,0.373059,-0.217053,-0.219824,-1.465241,-1.382275,-0.379374,-0.423022
5-Fluorouracil,-0.251688,-0.593183,-0.964523,-2.081383,-1.375403,-1.828508,-1.402174,-0.302262,-0.606540,-0.542744,...,-2.074054,-1.214244,-2.029475,-2.443552,-1.769484,0.075533,-1.505270,-2.840643,-2.166070,-1.851402
A-443654,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.474425,NaN,NaN,...,0.245932,-0.965712,NaN,NaN,NaN,-0.972180,NaN,NaN,0.886166,0.840518
A-770041,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.261659,NaN,NaN,...,-1.655043,-1.570598,NaN,NaN,NaN,-1.668421,NaN,NaN,-0.102603,-1.223374
A-83-01,-2.301715,-2.332981,-1.647609,-1.535165,-1.671070,-2.010003,-1.989658,-1.500976,-1.674289,-1.404062,...,-1.811814,-2.057040,-1.720772,-1.859382,-1.906683,-1.863604,-1.600242,-2.292330,-1.765835,-2.136302
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZSTK474,0.248269,-0.022731,-0.970869,-1.024803,-0.104083,-1.053312,-0.262107,1.054755,0.178557,0.367441,...,-0.068715,-0.589706,-0.951531,-0.105156,-0.312222,1.487813,0.828550,-1.980894,-1.027757,-0.592702
Zibotentan,-2.556426,-2.353534,-2.386170,-2.380730,-2.234259,-2.234084,-2.499997,-2.095787,-2.363119,-1.949842,...,-2.246525,-2.258756,-2.672580,-2.242702,-2.192390,-2.243488,-2.838531,-2.637342,-2.527769,-2.698189
"eEF2K Inhibitor, A-484954",-2.558452,-2.371571,-2.417482,-2.445182,-2.290205,-2.233704,-2.358586,-2.037009,-2.428601,-2.211754,...,-2.208470,-2.168980,-2.839630,-2.405680,-2.432065,-2.263309,-2.429701,-2.457831,-2.607461,-2.633146
kb NB 142-70,-1.298391,-1.297777,-1.338623,-1.154106,-1.094135,-0.859295,-1.187140,-0.297311,-0.794760,-0.851473,...,-1.359482,-1.268167,-1.574661,-0.699414,-0.869945,-0.703808,-1.507726,-2.636119,-1.273539,-1.218456


In [13]:
df = pd.DataFrame()
for i in drug_response.columns:
    tmp = pd.DataFrame(drug_response[i]).reset_index().dropna()
    tmp["cell"] = [i] * len(tmp)
    tmp.columns = ["Drug", "Value", "Cell"]
    tmp = tmp[["Drug", "Cell", "Value"]]
    df = pd.concat([df, tmp])
df

,Drug,Cell,Value
0,(5Z)-7-Oxozeaenol,22RV1,-1.074268
1,5-Fluorouracil,22RV1,-0.251688
4,A-83-01,22RV1,-2.301715
5,ACY-1215,22RV1,-0.368522
6,AGI-6780,22RV1,-0.900229
...,...,...,...
325,ZM447439,no-11,-1.050465
326,ZSTK474,no-11,-0.592702
327,Zibotentan,no-11,-2.698189
328,"eEF2K Inhibitor, A-484954",no-11,-2.633146


In [14]:
np.sum(df.Value > 0)

51189

In [15]:
np.sum(df.Value < 0)

209915

# Zero shot prediction

In [16]:
unique_drugs = df["Drug"].unique()
unique_cells = df["Cell"].unique()

# Split drugs and cell lines into training, validation, and test sets
train_drugs, test_val_drugs = train_test_split(
    unique_drugs, test_size=0.5, random_state=42
)
val_drugs, test_drugs = train_test_split(test_val_drugs, test_size=0.5, random_state=42)

train_cells, test_val_cells = train_test_split(
    unique_cells, test_size=0.55, random_state=42
)
val_cells, test_cells = train_test_split(test_val_cells, test_size=0.5, random_state=42)

# Split the dataset
train_df = df[(df["Drug"].isin(train_drugs)) & (df["Cell"].isin(train_cells))]
val_df = df[(df["Drug"].isin(val_drugs)) & (df["Cell"].isin(val_cells))]
test_df = df[(df["Drug"].isin(test_drugs)) & (df["Cell"].isin(test_cells))]


# Function to balance label distribution
def balance_labels(df, threshold=0.5):
    positive = df[df["Value"] > 0]
    negative = df[df["Value"] <= 0]
    min_count = min(len(positive), len(negative))
    balanced_positive = positive.sample(min_count, random_state=42)
    balanced_negative = negative.sample(min_count, random_state=42)
    return pd.concat([balanced_positive, balanced_negative])


# Balance label distribution across all sets
train_df = balance_labels(train_df)
val_df = balance_labels(val_df)
test_df = balance_labels(test_df)

# Separate features and labels
X_train = train_df[["Drug", "Cell"]]
y_train = np.array(train_df["Value"] > 0, dtype=float)

X_val = val_df[["Drug", "Cell"]]
y_val = np.array(val_df["Value"] > 0, dtype=float)

X_test = test_df[["Drug", "Cell"]]
y_test = np.array(test_df["Value"] > 0, dtype=float)

# Calculate total samples
total_samples = len(X_train) + len(X_val) + len(X_test)


# Function to calculate and format label ratios
def get_label_ratio(y):
    unique, counts = np.unique(y, return_counts=True)
    total = len(y)
    ratio_str = ", ".join(
        [f"Label {label}: {count/total:.2%}" for label, count in zip(unique, counts)]
    )
    return ratio_str


# Display results with percentages and label ratios
print("Train:")
print(X_train.shape, y_train.shape)
print(f"Percentage: {len(X_train)/total_samples:.2%}")
print(f"Label Ratio: {get_label_ratio(y_train)}")

print("\nValidation:")
print(X_val.shape, y_val.shape)
print(f"Percentage: {len(X_val)/total_samples:.2%}")
print(f"Label Ratio: {get_label_ratio(y_val)}")

print("\nTest:")
print(X_test.shape, y_test.shape)
print(f"Percentage: {len(X_test)/total_samples:.2%}")
print(f"Label Ratio: {get_label_ratio(y_test)}")

print(f"\nTotal samples: {total_samples}")
print(
    f"Ratio (Train:Validation:Test): {len(X_train):.0f}:{len(X_val):.0f}:{len(X_test):.0f}"
)
print(
    f"Overall Label Ratio: {get_label_ratio(np.concatenate([y_train, y_val, y_test]))}"
)

# X_train.to_csv("../nci_data/train.csv", index=False)
# X_test.to_csv("../nci_data/test.csv", index=False)
# X_val.to_csv("../nci_data/val.csv", index=False)

# np.save("../nci_data/train_labels.npy", y_train)
# np.save("../nci_data/test_labels.npy", y_test)
# np.save("../nci_data/val_labels.npy", y_val)

Train:
(19604, 2) (19604,)
Percentage: 54.72%
Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%

Validation:
(8590, 2) (8590,)
Percentage: 23.98%
Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%

Test:
(7632, 2) (7632,)
Percentage: 21.30%
Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%

Total samples: 35826
Ratio (Train:Validation:Test): 19604:8590:7632
Overall Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%


In [17]:
drug_response = drug_response.fillna(0)
drug_response

,22RV1,23132-87,42-MG-BA,451Lu,5637,639-V,647-V,697,769-P,786-0,...,WSU-DLCL2,WSU-NHL,YAPC,YH-13,YKG-1,YT,ZR-75-30,huH-1,no-10,no-11
(5Z)-7-Oxozeaenol,-1.074268,-0.565177,-0.255724,1.441198,-0.706426,-0.341020,-0.247842,0.262330,-0.265945,-0.334071,...,-1.567757,0.035238,-1.205127,0.373059,-0.217053,-0.219824,-1.465241,-1.382275,-0.379374,-0.423022
5-Fluorouracil,-0.251688,-0.593183,-0.964523,-2.081383,-1.375403,-1.828508,-1.402174,-0.302262,-0.606540,-0.542744,...,-2.074054,-1.214244,-2.029475,-2.443552,-1.769484,0.075533,-1.505270,-2.840643,-2.166070,-1.851402
A-443654,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.474425,0.000000,0.000000,...,0.245932,-0.965712,0.000000,0.000000,0.000000,-0.972180,0.000000,0.000000,0.886166,0.840518
A-770041,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.261659,0.000000,0.000000,...,-1.655043,-1.570598,0.000000,0.000000,0.000000,-1.668421,0.000000,0.000000,-0.102603,-1.223374
A-83-01,-2.301715,-2.332981,-1.647609,-1.535165,-1.671070,-2.010003,-1.989658,-1.500976,-1.674289,-1.404062,...,-1.811814,-2.057040,-1.720772,-1.859382,-1.906683,-1.863604,-1.600242,-2.292330,-1.765835,-2.136302
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZSTK474,0.248269,-0.022731,-0.970869,-1.024803,-0.104083,-1.053312,-0.262107,1.054755,0.178557,0.367441,...,-0.068715,-0.589706,-0.951531,-0.105156,-0.312222,1.487813,0.828550,-1.980894,-1.027757,-0.592702
Zibotentan,-2.556426,-2.353534,-2.386170,-2.380730,-2.234259,-2.234084,-2.499997,-2.095787,-2.363119,-1.949842,...,-2.246525,-2.258756,-2.672580,-2.242702,-2.192390,-2.243488,-2.838531,-2.637342,-2.527769,-2.698189
"eEF2K Inhibitor, A-484954",-2.558452,-2.371571,-2.417482,-2.445182,-2.290205,-2.233704,-2.358586,-2.037009,-2.428601,-2.211754,...,-2.208470,-2.168980,-2.839630,-2.405680,-2.432065,-2.263309,-2.429701,-2.457831,-2.607461,-2.633146
kb NB 142-70,-1.298391,-1.297777,-1.338623,-1.154106,-1.094135,-0.859295,-1.187140,-0.297311,-0.794760,-0.851473,...,-1.359482,-1.268167,-1.574661,-0.699414,-0.869945,-0.703808,-1.507726,-2.636119,-1.273539,-1.218456


# Masking

In [18]:
# Validation Mask
for _, row in X_val.iterrows():
    drug_response.loc[row["Drug"], row["Cell"]] = 0

# Test Mask
for _, row in X_test.iterrows():
    drug_response.loc[row["Drug"], row["Cell"]] = 0

# DTI

In [19]:
dti = pd.read_csv("../data/full_table.csv")
dti = dti.dropna(subset="Drug Name").reset_index(drop=True)
dti.head()

,Drug Name,DrugBank ID,PubChem CID,PubChem SID,SMILES,Gene,NSC
0,Lepirudin,DB00001,NaN,46507011.0,NaN,F2,NaN
1,Cetuximab,DB00002,NaN,46507042.0,NaN,EGFR,NaN
2,Cetuximab,DB00002,NaN,46507042.0,NaN,FCGR3B,NaN
3,Cetuximab,DB00002,NaN,46507042.0,NaN,C1QA,NaN
4,Cetuximab,DB00002,NaN,46507042.0,NaN,C1QB,NaN


In [20]:
dti = dti[dti["Drug Name"].isin(drug_response.index)]
dti.head()

,Drug Name,DrugBank ID,PubChem CID,PubChem SID,SMILES,Gene,NSC
1061,Bortezomib,DB00188,387447.0,46508736.0,CC(C)C[C@H](NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)C1=...,PSMB5,681239.0
1062,Bortezomib,DB00188,387447.0,46508736.0,CC(C)C[C@H](NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)C1=...,PSMB1,681239.0
1138,Pyrimethamine,DB00205,4993.0,46505987.0,CCC1=C(C(N)=NC(N)=N1)C1=CC=C(Cl)C=C1,DHFR,3061.0
1139,Pyrimethamine,DB00205,4993.0,46505987.0,CCC1=C(C(N)=NC(N)=N1)C1=CC=C(Cl)C=C1,DHFR,757306.0
1140,Pyrimethamine,DB00205,4993.0,46505987.0,CCC1=C(C(N)=NC(N)=N1)C1=CC=C(Cl)C=C1,HEXB,3061.0


In [21]:
print("unique drugs:", len(set(dti["Drug Name"])))
print("unique genes:", len(set(dti.Gene)))

unique drugs: 74
unique genes: 172


In [22]:
len(set(drug_response.index) & set(dti["Drug Name"]))

74

In [23]:
len(set(gene_exp.columns) & set(dti.Gene))

166

## All drugs are in drug response.

In [24]:
dti = dti[dti.Gene.isin(set(gene_exp.columns) & set(dti.Gene))]
dti

,Drug Name,DrugBank ID,PubChem CID,PubChem SID,SMILES,Gene,NSC
1061,Bortezomib,DB00188,387447.0,46508736.0,CC(C)C[C@H](NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)C1=...,PSMB5,681239.0
1062,Bortezomib,DB00188,387447.0,46508736.0,CC(C)C[C@H](NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)C1=...,PSMB1,681239.0
1140,Pyrimethamine,DB00205,4993.0,46505987.0,CCC1=C(C(N)=NC(N)=N1)C1=CC=C(Cl)C=C1,HEXB,3061.0
1141,Pyrimethamine,DB00205,4993.0,46505987.0,CCC1=C(C(N)=NC(N)=N1)C1=CC=C(Cl)C=C1,HEXB,757306.0
1454,Bleomycin,DB00290,5360373.0,46509116.0,[H][C@](C)(NC(=O)[C@@]([H])(NC(=O)C1=NC(=NC(N)...,LIG1,NaN
...,...,...,...,...,...,...,...
19475,Amuvatinib,DB12742,11282283.0,347828932.0,S=C(NCC1=CC=C2OCOC2=C1)N1CCN(CC1)C1=NC=NC2=C1O...,RET,NaN
19476,Amuvatinib,DB12742,11282283.0,347828932.0,S=C(NCC1=CC=C2OCOC2=C1)N1CCN(CC1)C1=NC=NC2=C1O...,PDGFRA,NaN
19477,Amuvatinib,DB12742,11282283.0,347828932.0,S=C(NCC1=CC=C2OCOC2=C1)N1CCN(CC1)C1=NC=NC2=C1O...,FLT3,NaN
19478,Amuvatinib,DB12742,11282283.0,347828932.0,S=C(NCC1=CC=C2OCOC2=C1)N1CCN(CC1)C1=NC=NC2=C1O...,RAD51,NaN


# Selected highly variable genes

In [25]:
variance = gene_exp.std()
variance = variance.sort_values(ascending=False)
variance = pd.DataFrame(variance > np.percentile(variance, 90))
variance = list(variance[variance[0]].index)
len(variance)

1957

In [26]:
print("DTI unique genes: ", len(set(dti["Gene"])))
print("Top 90% variable genes: ", len(variance))
print("Total: ", len(set(variance) | (set(dti["Gene"]))))

DTI unique genes:  166
Top 90% variable genes:  1957
Total:  2099


# Preprocessed data dims

In [27]:
genes = sorted(list(set(variance) | (set(dti["Gene"]))))
gene_exp = gene_exp[genes]
gene_exp.shape

(916, 2099)

In [28]:
drug_response.shape

(331, 916)

# Normalize

In [29]:
gene_norm_cell = pd.DataFrame(
    StandardScaler().fit_transform(gene_exp),
    index=gene_exp.index,
    columns=gene_exp.columns,
)
gene_norm_cell

Gene,A2M,AAED1,ABCC3,ABCG1,ABHD17C,ABI3BP,ABL1,ABL2,ABLIM1,ACAP1,...,ZNF462,ZNF467,ZNF608,ZNF667-AS1,ZNF711,ZNF726,ZNF83,ZNF853,ZP3,ZSCAN31
22RV1,0.799992,-0.369759,-0.891166,-0.117135,0.356264,-0.573802,-0.380785,-0.854328,1.302254,-0.519825,...,-0.441916,-0.220043,-0.572010,0.921612,-0.035489,2.432321,0.718230,-0.946038,0.762681,0.086164
23132-87,-0.292628,0.435610,1.797484,2.444006,1.697706,-0.471778,-0.911179,-0.609818,1.833529,-0.539513,...,-1.159331,-0.151843,-0.830986,-0.772851,-0.931668,-0.797771,0.246179,-0.794481,1.238434,0.053056
42-MG-BA,-0.432371,0.313453,0.167298,-0.084749,-0.642632,-0.016291,0.908722,-0.214799,-1.644293,-0.221937,...,0.571470,-0.549650,0.971135,1.951396,0.802608,-0.591467,0.643378,0.102898,-1.275833,3.110998
451Lu,4.016180,-0.405061,-0.980068,-0.827938,0.263072,-0.397448,-1.234196,0.900099,-1.647736,-0.651795,...,0.252867,-0.978947,0.756908,-0.763174,-0.900153,-0.098832,-2.447026,-0.424246,-0.126641,-1.096985
5637,-0.472501,-0.555879,0.502792,-0.012260,0.008421,-0.385946,1.494821,0.292600,0.337625,-0.393989,...,1.641429,-0.863154,0.699903,-0.456158,0.169265,0.507041,-0.510085,1.640649,2.239234,0.160734
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YT,0.659608,-2.889206,-0.788713,-0.811647,0.131661,1.247839,-1.752790,-1.330305,0.376665,2.586523,...,-1.363830,-0.829363,-1.213770,-0.785057,-0.606639,0.085774,0.688913,-0.633507,-0.970403,-1.278730
ZR-75-30,-0.247356,-0.975722,1.475665,2.172460,1.217535,-0.569815,-0.933282,0.333823,0.344540,-0.470265,...,-0.511466,1.773184,-0.932873,-0.754765,-0.919524,-0.590822,0.586915,0.309848,0.823979,-0.135636
huH-1,0.632912,0.953757,1.881583,-0.946129,-0.634827,-0.550759,0.299767,-1.719611,0.937043,-0.572606,...,-0.932128,-1.077578,-1.109602,0.452248,1.273927,1.705354,-1.210227,-1.016432,0.130505,2.389444
no-10,-0.538384,0.684730,0.466495,-0.888639,-0.529349,4.138973,-0.303512,1.090892,-0.264811,-0.556646,...,0.086847,-1.028898,0.363480,0.812418,0.806279,0.021787,0.827230,-0.428582,-1.431527,-0.568603


In [30]:
gene_norm_gene = pd.DataFrame(
    StandardScaler().fit_transform(gene_exp.T.values),
    index=gene_exp.columns,
    columns=gene_exp.index,
).T
gene_norm_gene

Gene,A2M,AAED1,ABCC3,ABCG1,ABHD17C,ABI3BP,ABL1,ABL2,ABLIM1,ACAP1,...,ZNF462,ZNF467,ZNF608,ZNF667-AS1,ZNF711,ZNF726,ZNF83,ZNF853,ZP3,ZSCAN31
22RV1,0.087426,0.400890,-0.943254,-0.446177,0.960113,-1.045388,0.026530,-0.171508,1.534299,-0.972916,...,0.200542,-0.231790,-0.571361,0.798981,-0.377768,1.297097,1.588874,-1.037361,0.596610,-0.294466
23132-87,-0.743876,0.728210,0.940470,0.982471,1.607997,-0.953537,-0.180718,-0.156223,1.679049,-0.960007,...,-0.342422,-0.233388,-0.730010,-0.892782,-0.999285,-0.615516,1.023597,-0.920425,0.735384,-0.352340
42-MG-BA,-0.932402,0.531987,-0.297622,-0.555477,0.082875,-0.776183,0.168114,-0.160333,-0.754979,-0.877929,...,0.625275,-0.593290,0.155716,1.442524,0.023883,-0.606035,1.168905,-0.507945,-0.750144,1.142407
451Lu,2.240135,0.255481,-1.021787,-0.912123,0.731380,-0.945838,-0.292279,0.236209,-0.694231,-1.060488,...,0.538615,-0.806999,0.134249,-0.920516,-1.016424,-0.270653,-0.998463,-0.741888,-0.043481,-1.007388
5637,-1.083064,-0.016944,-0.169094,-0.626156,0.381618,-1.116958,0.228778,-0.121533,0.462650,-1.094496,...,1.229639,-0.910971,-0.087495,-0.817531,-0.488776,-0.132142,0.254827,0.267237,1.078705,-0.507970
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YT,0.110118,-0.934930,-0.711945,-0.728185,0.918183,0.286588,-0.225211,-0.177513,0.953626,1.025671,...,-0.305205,-0.522171,-0.811253,-0.768485,-0.641390,0.047154,1.651466,-0.686877,-0.318326,-0.943790
ZR-75-30,-0.703621,0.012672,0.798004,0.914295,1.430044,-1.018419,-0.151143,0.151942,0.756846,-0.923251,...,0.117680,1.115677,-0.781662,-0.875476,-0.996279,-0.485811,1.383025,-0.255603,0.576392,-0.431710
huH-1,-0.110378,1.018420,1.010300,-0.962574,0.205044,-1.021683,0.129442,-0.472177,1.089351,-0.998659,...,-0.205695,-0.854234,-0.904471,0.226022,0.453660,0.726475,-0.054897,-1.069448,0.120832,0.911760
no-10,-1.012602,0.796771,-0.062924,-1.008916,0.193140,1.766718,-0.105917,0.233799,0.192231,-1.069378,...,0.376975,-0.901256,-0.145785,0.484988,0.064547,-0.263420,1.397512,-0.805204,-0.834967,-0.780587


# Make matrices association matrices by setting 0 threshold and min max scaler.

In [31]:
def min_max_scale(data):
    scaler = MinMaxScaler(feature_range=(0, 1))
    data = data[data > 0].fillna(0)
    return pd.DataFrame(
        scaler.fit_transform(data), index=data.index, columns=data.columns
    )

In [32]:
A_dc = min_max_scale(drug_response)
A_dc

,22RV1,23132-87,42-MG-BA,451Lu,5637,639-V,647-V,697,769-P,786-0,...,WSU-DLCL2,WSU-NHL,YAPC,YH-13,YKG-1,YT,ZR-75-30,huH-1,no-10,no-11
(5Z)-7-Oxozeaenol,0.000000,0.0,0.0,0.540895,0.0,0.0,0.0,0.083315,0.00000,0.000000,...,0.000000,0.013851,0.0,0.125799,0.0,0.000000,0.000000,0.0,0.000000,0.000000
5-Fluorouracil,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.00000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.0,0.025421,0.000000,0.0,0.000000,0.000000
A-443654,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.468272,0.00000,0.000000,...,0.090899,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.345059,0.300918
A-770041,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.00000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000
A-83-01,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.00000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZSTK474,0.099818,0.0,0.0,0.000000,0.0,0.0,0.0,0.334986,0.05717,0.121001,...,0.000000,0.000000,0.0,0.000000,0.0,0.500727,0.402895,0.0,0.000000,0.000000
Zibotentan,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.00000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000
"eEF2K Inhibitor, A-484954",0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.00000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000
kb NB 142-70,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.00000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000


In [33]:
A_cg = min_max_scale(gene_norm_gene + gene_norm_cell)
A_cg

Gene,A2M,AAED1,ABCC3,ABCG1,ABHD17C,ABI3BP,ABL1,ABL2,ABLIM1,ACAP1,...,ZNF462,ZNF467,ZNF608,ZNF667-AS1,ZNF711,ZNF726,ZNF83,ZNF853,ZP3,ZSCAN31
22RV1,0.101248,0.008050,0.000000,0.000000,0.275005,0.000000,0.000000,0.000000,0.483687,0.000000,...,0.000000,0.00000,0.000000,0.362624,0.000000,0.792601,0.575311,0.000000,0.239847,0.000000
23132-87,0.000000,0.300948,0.643935,0.731081,0.690597,0.000000,0.000000,0.000000,0.598963,0.000000,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.316638,0.000000,0.348281,0.000000
42-MG-BA,0.000000,0.218619,0.000000,0.000000,0.000000,0.000000,0.243516,0.000000,0.000000,0.000000,...,0.307132,0.00000,0.228247,0.715287,0.187764,0.000000,0.451920,0.000000,0.000000,0.733755
451Lu,0.713800,0.000000,0.000000,0.000000,0.207752,0.000000,0.000000,0.231809,0.000000,0.000000,...,0.203125,0.00000,0.180506,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5637,0.000000,0.000000,0.078482,0.000000,0.081483,0.000000,0.389775,0.034898,0.136462,0.000000,...,0.736829,0.00000,0.124045,0.000000,0.000000,0.079676,0.000000,0.407554,0.585451,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YT,0.087820,0.000000,0.000000,0.000000,0.219324,0.207919,0.000000,0.000000,0.226840,0.637472,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.028251,0.583609,0.000000,0.000000,0.000000
ZR-75-30,0.000000,0.000000,0.534741,0.658597,0.553107,0.000000,0.000000,0.099097,0.187808,0.000000,...,0.000000,0.56017,0.000000,0.000000,0.000000,0.000000,0.491234,0.011588,0.247096,0.000000
huH-1,0.059617,0.509977,0.680138,0.000000,0.000000,0.000000,0.097061,0.000000,0.345539,0.000000,...,0.000000,0.00000,0.000000,0.142949,0.392476,0.516829,0.000000,0.000000,0.044348,0.569490
no-10,0.000000,0.383095,0.094915,0.000000,0.000000,0.800238,0.000000,0.270239,0.000000,0.000000,...,0.119035,0.00000,0.044095,0.273436,0.197836,0.000000,0.554773,0.000000,0.000000,0.000000


In [34]:
A_dg = (
    pd.DataFrame(
        np.ones([len(A_dc.index), len(A_cg.columns)]),
        index=A_dc.index,
        columns=A_cg.columns,
    )
    / 2
)
for _, i in dti.iterrows():
    A_dg.loc[i["Drug Name"], i["Gene"]] = 1
A_dg

Gene,A2M,AAED1,ABCC3,ABCG1,ABHD17C,ABI3BP,ABL1,ABL2,ABLIM1,ACAP1,...,ZNF462,ZNF467,ZNF608,ZNF667-AS1,ZNF711,ZNF726,ZNF83,ZNF853,ZP3,ZSCAN31
(5Z)-7-Oxozeaenol,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
5-Fluorouracil,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
A-443654,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
A-770041,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
A-83-01,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZSTK474,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Zibotentan,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
"eEF2K Inhibitor, A-484954",0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
kb NB 142-70,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5


In [35]:
A_dg.sum().sum()

347531.5

In [36]:
print(
    "Drug Density: ",
    round(len(A_dc.values.nonzero()[0]) / drug_response.size, 4) * 100,
    "%",
)
print("Cell Density: ", round(len(A_cg.values.nonzero()[0]) / A_cg.size, 4) * 100, "%")
print("Gene Density: ", round(len(A_dg.values.nonzero()[0]) / A_dg.size, 5) * 100, "%")

Drug Density:  14.21 %
Cell Density:  41.8 %
Gene Density:  100.0 %


# Similarity

In [37]:
def normalize_similarity_matrix(df, gamma=None):
    similarity_matrix = rbf_kernel(df.values, gamma=gamma)
    scaler = MinMaxScaler()
    normalized_matrix = scaler.fit_transform(similarity_matrix.reshape(-1, 1))
    normalized_df = pd.DataFrame(
        normalized_matrix.reshape(similarity_matrix.shape),
        index=df.index,
        columns=df.index,
    )

    return normalized_df

In [38]:
cell_sim = normalize_similarity_matrix(drug_response.T)
cell_sim.to_csv("../gdsc1_data/cell_sim.csv")
cell_sim

,22RV1,23132-87,42-MG-BA,451Lu,5637,639-V,647-V,697,769-P,786-0,...,WSU-DLCL2,WSU-NHL,YAPC,YH-13,YKG-1,YT,ZR-75-30,huH-1,no-10,no-11
22RV1,1.000000,0.577971,0.453663,0.418010,0.547923,0.473040,0.457391,0.387181,0.516971,0.582887,...,0.356693,0.363672,0.435953,0.539988,0.631277,0.363698,0.431897,0.323181,0.466738,0.409974
23132-87,0.577971,1.000000,0.637920,0.515960,0.756751,0.655063,0.654736,0.354096,0.714857,0.619226,...,0.364099,0.519946,0.507691,0.713109,0.646245,0.430206,0.455894,0.381204,0.466671,0.458464
42-MG-BA,0.453663,0.637920,1.000000,0.583295,0.641596,0.709764,0.665469,0.279696,0.555451,0.490645,...,0.384780,0.581042,0.498336,0.671954,0.555017,0.423944,0.343512,0.398174,0.434244,0.429820
451Lu,0.418010,0.515960,0.583295,1.000000,0.507786,0.605584,0.580609,0.204703,0.475141,0.397434,...,0.272288,0.433457,0.403917,0.549824,0.447723,0.373999,0.355075,0.363341,0.387001,0.417217
5637,0.547923,0.756751,0.641596,0.507786,1.000000,0.686497,0.610384,0.417377,0.743279,0.666430,...,0.338396,0.519559,0.431348,0.712475,0.639099,0.418440,0.415355,0.302727,0.401998,0.425982
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YT,0.363698,0.430206,0.423944,0.373999,0.418440,0.422069,0.406313,0.295477,0.400136,0.426755,...,0.342535,0.556613,0.254565,0.403514,0.373767,1.000000,0.319061,0.172611,0.338024,0.395492
ZR-75-30,0.431897,0.455894,0.343512,0.355075,0.415355,0.392543,0.381332,0.149997,0.353136,0.369816,...,0.228549,0.280084,0.403288,0.417435,0.396405,0.319061,1.000000,0.259403,0.395145,0.417089
huH-1,0.323181,0.381204,0.398174,0.363341,0.302727,0.374472,0.393032,0.086102,0.256757,0.246709,...,0.311266,0.277964,0.549027,0.413513,0.371956,0.172611,0.259403,1.000000,0.391980,0.362260
no-10,0.466738,0.466671,0.434244,0.387001,0.401998,0.443862,0.475733,0.200564,0.395398,0.399011,...,0.415779,0.344564,0.518710,0.453506,0.490323,0.338024,0.395145,0.391980,1.000000,0.583804


In [39]:
print("Min:", np.min(cell_sim.values))
print("Max:", np.max(cell_sim.values))
print("Mean:", np.mean(cell_sim.values))
print("Median:", np.median(cell_sim.values))

Min: 0.0
Max: 1.0
Mean: 0.3990938722640615
Median: 0.4026713949796482


In [40]:
gene_sim = normalize_similarity_matrix(gene_norm_cell.T)
gene_sim.to_csv("../gdsc1_data/gene_sim.csv")
gene_sim

Gene,A2M,AAED1,ABCC3,ABCG1,ABHD17C,ABI3BP,ABL1,ABL2,ABLIM1,ACAP1,...,ZNF462,ZNF467,ZNF608,ZNF667-AS1,ZNF711,ZNF726,ZNF83,ZNF853,ZP3,ZSCAN31
Gene,,,,,,,,,,,,,,,,,,,,,
A2M,1.000000,0.124485,0.089056,0.073519,0.106712,0.123790,0.117995,0.182122,0.081755,0.089229,...,0.140772,0.092125,0.159805,0.132448,0.083841,0.115005,0.099288,0.130699,0.079870,0.089537
AAED1,0.124485,1.000000,0.167643,0.065413,0.140633,0.181263,0.175015,0.218211,0.085385,0.045949,...,0.208434,0.113114,0.079866,0.129671,0.058985,0.064748,0.098490,0.097863,0.172413,0.153373
ABCC3,0.089056,0.167643,1.000000,0.132297,0.248288,0.163086,0.109223,0.112212,0.249072,0.045986,...,0.154874,0.126112,0.095569,0.076564,0.047226,0.052629,0.104062,0.084536,0.192772,0.230415
ABCG1,0.073519,0.065413,0.132297,1.000000,0.157374,0.079918,0.092199,0.057071,0.187779,0.141018,...,0.079611,0.171769,0.097116,0.083606,0.166331,0.133404,0.112871,0.131236,0.119962,0.146983
ABHD17C,0.106712,0.140633,0.248288,0.157374,1.000000,0.111474,0.073673,0.107191,0.210761,0.051542,...,0.152614,0.095619,0.069326,0.061393,0.053965,0.053513,0.069275,0.099703,0.191379,0.150871
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF726,0.115005,0.064748,0.052629,0.133404,0.053513,0.089044,0.114655,0.088333,0.085117,0.183387,...,0.097739,0.119783,0.154556,0.210074,0.228810,1.000000,0.189290,0.148906,0.072056,0.097355
ZNF83,0.099288,0.098490,0.104062,0.112871,0.069275,0.143355,0.158708,0.121674,0.089111,0.125705,...,0.117412,0.162111,0.170624,0.215666,0.141006,0.189290,1.000000,0.153252,0.096197,0.109052
ZNF853,0.130699,0.097863,0.084536,0.131236,0.099703,0.094121,0.153785,0.112366,0.124960,0.062891,...,0.195961,0.172315,0.181725,0.251559,0.268304,0.148906,0.153252,1.000000,0.080663,0.122058


In [41]:
print("Min:", np.min(gene_sim.values))
print("Max:", np.max(gene_sim.values))
print("Mean:", np.mean(gene_sim.values))
print("Median:", np.median(gene_sim.values))

Min: 0.0
Max: 1.0
Mean: 0.1336409172868509
Median: 0.11894801789771914


# NSC to SMILES

In [42]:
def get_morgan_fingerprint(SMILES):
    # Initialize parser parameters
    params = Chem.SmilesParserParams()
    params.useChirality = True  # Preserve stereochemistry information
    params.removeHs = False  # Keep hydrogen atoms
    mfp = []

    for smi in SMILES:
        mol = None
        # Sanitization attempt strategies
        sanitize_attempts = [
            {"sanitize": True},  # First try with standard sanitization
            {
                "sanitize": False,
                "partial_sanitize": True,
            },  # Fallback: partial sanitization
        ]

        for attempt in sanitize_attempts:
            try:
                # Update parameters for this attempt
                current_params = Chem.SmilesParserParams()
                current_params.sanitize = attempt["sanitize"]
                current_params.useChirality = params.useChirality
                current_params.removeHs = params.removeHs

                # Molecule object creation
                mol = Chem.MolFromSmiles(smi, params=current_params)

                if mol and "partial_sanitize" in attempt:
                    # Perform partial sanitization (skip property validation)
                    Chem.SanitizeMol(mol, Chem.SANITIZE_ALL ^ Chem.SANITIZE_PROPERTIES)

                if mol:  # Successfully processed molecule
                    # Generate Morgan fingerprint
                    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
                    mfp.append(np.array(fp))
                    break  # Exit attempt loop on success

            except Exception as e:
                if attempt == sanitize_attempts[-1]:  # Final attempt failed
                    print(f"Processing failed: {smi}")
                    print(f"Error details: {str(e)}")
                continue  # Try next attempt

        if not mol:  # All attempts failed
            print(f"Complete processing failure: {smi}")
            mfp.append(np.zeros(2048))  # Insert zero-vector placeholder

    return np.array(mfp)

In [43]:
conv = dict(SMILES[["drugs", "SMILES"]].values)
mfp = get_morgan_fingerprint([conv[i] for i in drug_response.index])
mfp = pd.DataFrame(mfp, index=drug_response.index)

In [44]:
drug_sim = normalize_similarity_matrix(mfp)
drug_sim.to_csv("../gdsc1_data/drug_sim.csv")
drug_sim

,(5Z)-7-Oxozeaenol,5-Fluorouracil,A-443654,A-770041,A-83-01,ACY-1215,AGI-6780,AICA Ribonucleotide,AKT inhibitor VIII_171,AKT inhibitor VIII_228,...,YK-4-279,YM201636,Z-LLNle-CHO,ZG-10,ZM447439,ZSTK474,Zibotentan,"eEF2K Inhibitor, A-484954",kb NB 142-70,torin2
(5Z)-7-Oxozeaenol,1.000000,0.724363,0.567366,0.523651,0.562500,0.616165,0.562500,0.625953,0.552773,0.552773,...,0.679960,0.533349,0.606386,0.557635,0.591736,0.635751,0.596617,0.655375,0.679960,0.567366
5-Fluorouracil,0.724363,1.000000,0.679960,0.557635,0.665202,0.729309,0.655375,0.709540,0.655375,0.655375,...,0.754074,0.606386,0.709540,0.630851,0.616165,0.749116,0.660287,0.729309,0.773930,0.699671
A-443654,0.567366,0.679960,1.000000,0.480128,0.577107,0.611274,0.489783,0.562500,0.577107,0.577107,...,0.606386,0.509122,0.572236,0.552773,0.538202,0.611274,0.562500,0.581981,0.596617,0.630851
A-770041,0.523651,0.557635,0.480128,1.000000,0.513963,0.538202,0.504284,0.528499,0.543057,0.543057,...,0.523651,0.494614,0.509122,0.480128,0.562500,0.557635,0.518806,0.596617,0.523651,0.499448
A-83-01,0.562500,0.665202,0.577107,0.513963,1.000000,0.625953,0.562500,0.577107,0.562500,0.562500,...,0.601500,0.591736,0.596617,0.616165,0.591736,0.655375,0.606386,0.606386,0.630851,0.596617
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZSTK474,0.635751,0.749116,0.611274,0.557635,0.655375,0.660287,0.586857,0.640653,0.635751,0.635751,...,0.645558,0.645558,0.640653,0.581981,0.645558,1.000000,0.621058,0.660287,0.675038,0.660287
Zibotentan,0.596617,0.660287,0.562500,0.518806,0.606386,0.611274,0.557635,0.572236,0.557635,0.557635,...,0.606386,0.557635,0.572236,0.562500,0.557635,0.621058,1.000000,0.591736,0.606386,0.572236
"eEF2K Inhibitor, A-484954",0.655375,0.729309,0.581981,0.596617,0.606386,0.650466,0.586857,0.699671,0.635751,0.635751,...,0.665202,0.586857,0.621058,0.581981,0.586857,0.660287,0.591736,1.000000,0.684884,0.640653
kb NB 142-70,0.679960,0.773930,0.596617,0.523651,0.630851,0.665202,0.611274,0.635751,0.591736,0.591736,...,0.689810,0.581981,0.635751,0.606386,0.581981,0.675038,0.606386,0.684884,1.000000,0.625953


In [45]:
print("Min:", np.min(drug_sim.values))
print("Max:", np.max(drug_sim.values))
print("Mean:", np.mean(drug_sim.values))
print("Median:", np.median(drug_sim.values))

Min: 0.0
Max: 1.0
Mean: 0.5621594450071402
Median: 0.5673664611068219


# Unified Graph

In [46]:
A_cg

Gene,A2M,AAED1,ABCC3,ABCG1,ABHD17C,ABI3BP,ABL1,ABL2,ABLIM1,ACAP1,...,ZNF462,ZNF467,ZNF608,ZNF667-AS1,ZNF711,ZNF726,ZNF83,ZNF853,ZP3,ZSCAN31
22RV1,0.101248,0.008050,0.000000,0.000000,0.275005,0.000000,0.000000,0.000000,0.483687,0.000000,...,0.000000,0.00000,0.000000,0.362624,0.000000,0.792601,0.575311,0.000000,0.239847,0.000000
23132-87,0.000000,0.300948,0.643935,0.731081,0.690597,0.000000,0.000000,0.000000,0.598963,0.000000,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.316638,0.000000,0.348281,0.000000
42-MG-BA,0.000000,0.218619,0.000000,0.000000,0.000000,0.000000,0.243516,0.000000,0.000000,0.000000,...,0.307132,0.00000,0.228247,0.715287,0.187764,0.000000,0.451920,0.000000,0.000000,0.733755
451Lu,0.713800,0.000000,0.000000,0.000000,0.207752,0.000000,0.000000,0.231809,0.000000,0.000000,...,0.203125,0.00000,0.180506,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5637,0.000000,0.000000,0.078482,0.000000,0.081483,0.000000,0.389775,0.034898,0.136462,0.000000,...,0.736829,0.00000,0.124045,0.000000,0.000000,0.079676,0.000000,0.407554,0.585451,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YT,0.087820,0.000000,0.000000,0.000000,0.219324,0.207919,0.000000,0.000000,0.226840,0.637472,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.028251,0.583609,0.000000,0.000000,0.000000
ZR-75-30,0.000000,0.000000,0.534741,0.658597,0.553107,0.000000,0.000000,0.099097,0.187808,0.000000,...,0.000000,0.56017,0.000000,0.000000,0.000000,0.000000,0.491234,0.011588,0.247096,0.000000
huH-1,0.059617,0.509977,0.680138,0.000000,0.000000,0.000000,0.097061,0.000000,0.345539,0.000000,...,0.000000,0.00000,0.000000,0.142949,0.392476,0.516829,0.000000,0.000000,0.044348,0.569490
no-10,0.000000,0.383095,0.094915,0.000000,0.000000,0.800238,0.000000,0.270239,0.000000,0.000000,...,0.119035,0.00000,0.044095,0.273436,0.197836,0.000000,0.554773,0.000000,0.000000,0.000000


In [47]:
indexes = list(A_dc.index) + list(A_cg.index) + list(A_dg.columns)
n_all = len(indexes)
base = pd.DataFrame(np.zeros([n_all, n_all]), index=indexes, columns=indexes)
base

,(5Z)-7-Oxozeaenol,5-Fluorouracil,A-443654,A-770041,A-83-01,ACY-1215,AGI-6780,AICA Ribonucleotide,AKT inhibitor VIII_171,AKT inhibitor VIII_228,...,ZNF462,ZNF467,ZNF608,ZNF667-AS1,ZNF711,ZNF726,ZNF83,ZNF853,ZP3,ZSCAN31
(5Z)-7-Oxozeaenol,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5-Fluorouracil,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A-443654,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A-770041,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A-83-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF726,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZNF83,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZNF853,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZP3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [48]:
base.loc[A_cg.index, A_cg.columns] = A_cg
base.loc[A_cg.columns, A_cg.index] = A_cg.T
base.loc[A_dc.index, A_dc.columns] = A_dc
base.loc[A_dc.columns, A_dc.index] = A_dc.T
base.loc[A_dg.index, A_dg.columns] = A_dg
base.loc[A_dg.columns, A_dg.index] = A_dg.T

In [49]:
base

,(5Z)-7-Oxozeaenol,5-Fluorouracil,A-443654,A-770041,A-83-01,ACY-1215,AGI-6780,AICA Ribonucleotide,AKT inhibitor VIII_171,AKT inhibitor VIII_228,...,ZNF462,ZNF467,ZNF608,ZNF667-AS1,ZNF711,ZNF726,ZNF83,ZNF853,ZP3,ZSCAN31
(5Z)-7-Oxozeaenol,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
5-Fluorouracil,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
A-443654,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
A-770041,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
A-83-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF726,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZNF83,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZNF853,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZP3,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [50]:
edge_index = np.array(base.values.nonzero())
edge_attr = np.array(base.values[base.values.nonzero()])

np.save(
    "../gdsc1_data/idxs.npy",
    pd.DataFrame([list(range(len(base.index))), base.index]).values,
)

np.save(
    "../gdsc1_data/edge_idxs.npy",
    edge_index,
)

np.save(
    "../gdsc1_data/edge_attr.npy",
    edge_attr,
)